# Reshape tables · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 19. Reshape tables

Move between long and wide layouts and choose when to aggregate.

Before you start: Understand keys, groups, and summary functions.

Official reference: https://pandas.pydata.org/docs/user_guide/reshaping.html

### M19C01. Melt wide columns into rows

A wide table stores different measures in different columns. melt turns selected measure columns into a label column and a value column. The ID is repeated so every new row still belongs to a student.

**Read:** Which column stays as an identifier?

Answer: Student_ID.

**Predict:** How many long rows will 3 students and 2 measures produce?

Answer: 6 rows.

**Explain:** Are Score and Attendance being added together?

Answer: No. melt changes the layout; the values remain separate and their measure labels are retained.



In [ ]:
# Predict before running: How many long rows will 3 students and 2 measures produce?
wide = students().head(3)[["Student_ID", "Score", "Attendance"]]
result = wide.melt(id_vars="Student_ID", value_vars=["Score", "Attendance"], var_name="Measure", value_name="Value")
print(result)

**Your turn:** Use the first four students and melt Score and Study_Hours instead.

For the check, keep these result names: `result`.

**Hint:** Change head(3) to head(4) and the two measure names.

**Watch for this:** A shared Value column may contain measures with different units; keep and use Measure.

In [ ]:
# Change this code to complete the task.
# Question: Use the first four students and melt Score and Study_Hours instead.
wide = students().head(4)[["Student_ID", "Score", "Study_Hours"]]
result = wide.melt(id_vars="Student_ID", var_name="Measure", value_name="Value")
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 8
assert set(result["Measure"]) == {"Score", "Study_Hours"}
print("Code checks passed. Review the written explanation as well.")

Four students with two measures produce eight rows.

**Quick check:** What does melt mainly change?

1. The values by averaging them
2. The table layout
3. The student IDs

Answer: 2. melt reorganizes values into a long layout without aggregating them.

**Explain what you learned:** Long form keeps one measured value per row.

### M19C02. Pivot a unique pair

pivot moves distinct category labels into columns. Every index-and-column pair must identify at most one value. Here each student has one Score and one Attendance value.

**Read:** Which labels become the output columns?

Answer: The distinct Measure values: Attendance and Score.

**Predict:** What is the Score for S002 after pivoting?

Answer: 72, unchanged.

**Explain:** Why can repeated Student_ID and Measure pairs cause an error?

Answer: pivot cannot choose which of several values belongs in the same output cell.



In [ ]:
# Predict before running: What is the Score for S002 after pivoting?
long = students().head(3).melt(id_vars="Student_ID", value_vars=["Score", "Attendance"], var_name="Measure", value_name="Value")
result = long.pivot(index="Student_ID", columns="Measure", values="Value")
print(result)

**Your turn:** Pivot the same long table so Measure labels form the rows and Student_ID labels form the columns.

For the check, keep these result names: `result`.

**Hint:** Exchange the arguments assigned to index and columns.

**Watch for this:** Use pivot_table only when you actually intend to aggregate repeated pairs.

In [ ]:
# Change this code to complete the task.
# Question: Pivot the same long table so Measure labels form the rows and Student_ID labels form the columns.
long = students().head(3).melt(id_vars="Student_ID", value_vars=["Score", "Attendance"], var_name="Measure", value_name="Value")
result = long.pivot(index="Measure", columns="Student_ID", values="Value")
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.shape == (2, 3)
assert result.loc["Score", "S002"] == 72
print("Code checks passed. Review the written explanation as well.")

The values stay the same, but the two table axes exchange roles.

**Quick check:** What must be unique for pivot?

1. Each row label alone
2. Each column label alone
3. Each index-and-column pair

Answer: 3. The pair identifies one output cell and must have a single value.

**Explain what you learned:** pivot rearranges uniquely identified values.

### M19C03. Aggregate with pivot_table

Several students share the same Campus and Program. pivot_table can reduce those repeated combinations using an explicit aggregation function. Naming aggfunc makes the summary rule clear.

**Read:** What does one cell summarize?

Answer: The mean Score of students with that Campus and Program.

**Predict:** How many campus-by-program cells are there?

Answer: Four: two campuses times two programs.

**Explain:** Why specify aggfunc?

Answer: It documents the statistic; mean, sum, and count answer different questions.



In [ ]:
# Predict before running: How many campus-by-program cells are there?
df = students()
result = df.pivot_table(index="Campus", columns="Program", values="Score", aggfunc="mean", observed=True)
print(result.round(2))

**Your turn:** Change the table to count Student_ID records for each campus and program.

For the check, keep these result names: `result`.

**Hint:** Use values="Student_ID" and aggfunc="count".

**Watch for this:** Filling an absent combination with zero implies a meaning; it is not automatically appropriate for a mean.

In [ ]:
# Change this code to complete the task.
# Question: Change the table to count Student_ID records for each campus and program.
df = students()
result = df.pivot_table(index="Campus", columns="Program", values="Student_ID", aggfunc="count", observed=True)
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.to_numpy().sum() == 12
assert result.loc["RAK", "AI"] == 4
print("Code checks passed. Review the written explanation as well.")

The counts are Dubai AI 2, Dubai IT 4, RAK AI 4, and RAK IT 2.

**Quick check:** Which method handles repeated pairs by summarizing them?

1. pivot_table
2. pivot without any changes
3. set_index alone

Answer: 1. pivot_table applies an aggregation to values sharing the same output cell.

**Explain what you learned:** Choose an aggregation that matches the question.

### M19C04. Count combinations with crosstab

crosstab counts how often two categories occur together. normalize="index" instead reports proportions within each row. The denominator changes, so always label the result.

**Read:** What is counted in each cell?

Answer: Students in that Campus and Program combination.

**Predict:** How many RAK students are in AI?

Answer: 4.

**Explain:** Would a larger count alone prove a stronger preference?

Answer: No. Group sizes matter; proportions may be more suitable for that question.



In [ ]:
# Predict before running: How many RAK students are in AI?
df = students()
result = pd.crosstab(df["Campus"], df["Program"])
print(result)

**Your turn:** Calculate program percentages within each campus and keep the unrounded percentages in result. Print a version rounded to one decimal place.

For the check, keep these result names: `result`.

**Hint:** Use normalize="index" and multiply by 100.

**Watch for this:** Say whether percentages are by row, by column, or of the whole table.

In [ ]:
# Change this code to complete the task.
# Question: Calculate program percentages within each campus and keep the unrounded percentages in result. Print a version rounded to one decimal place.
df = students()
result = pd.crosstab(df["Campus"], df["Program"], normalize="index") * 100
print(result.round(1))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert np.allclose(result.sum(axis=1), 100)
assert np.isclose(result.loc["Dubai", "AI"], 100 / 3)
print("Code checks passed. Review the written explanation as well.")

Each campus row sums to 100%. AI accounts for about 66.7% in RAK and 33.3% in Dubai in this fictional table.

**Quick check:** normalize="index" uses which denominator?

1. Each row total
2. The entire table total
3. Each column total

Answer: 1. Each cell is divided by its own row total.

**Explain what you learned:** Counts and proportions answer different questions.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates